# Week 4 — Pain Expression Segmentation with YOLOv8-seg

**Model:** YOLOv8n-seg | **Classes:** No_Pain · Mild · Moderate · Severe | **Metric:** Mask mAP@0.5

### Steps
1. Mount Drive → run Cell 1 manually first, then Run All for the rest
2. Install → Extract → Build dataset → Train → Evaluate → Save

In [ ]:
# ── STEP 1: Run this cell manually first, click the auth link, wait for 'Mounted' ──
from google.colab import drive
drive.mount('/content/drive', force_remount=True)
print('Drive mounted.')

import os
print('Files on Drive:')
for f in sorted(os.listdir('/content/drive/MyDrive/'))[:30]:
    print(' ', f)

In [ ]:
# ── STEP 2: Install ──────────────────────────────────────────────────────────
!pip install ultralytics -q
print('ultralytics installed.')

In [ ]:
# ── STEP 3: Setup — imports, config, extract, collect images ─────────────────
import zipfile, shutil, os, random, json
from pathlib import Path
from collections import Counter
import cv2
import numpy as np

# ── CHANGE THIS if your ZIP has a different name ──────────────────────────────
DRIVE_ZIP = '/content/drive/MyDrive/pain_dataset_new.zip'
# ─────────────────────────────────────────────────────────────────────────────

WORK_DIR    = Path('/content/pain_detection')
YOLO_DIR    = Path('/content/yolo_seg_dataset')
RESULTS_DIR = Path('/content/pain_seg_results')

CLASSES     = ['No_Pain', 'Mild', 'Moderate', 'Severe']
CLASS_TO_ID = {c: i for i, c in enumerate(CLASSES)}
ID_TO_CLASS = {i: c for i, c in enumerate(CLASSES)}
IMG_EXTS    = {'.jpg', '.jpeg', '.png', '.bmp'}
TRAIN_RATIO = 0.70
VAL_RATIO   = 0.15

EMOTION_TO_PAIN = {
    'neutral':'No_Pain','happiness':'No_Pain','hapiness':'No_Pain',
    'sadness':'Mild',
    'fear':'Moderate','surprise':'Moderate','suprise':'Moderate','surpris':'Moderate',
    'anger':'Severe','disgust':'Severe','disgest':'Severe',
}

COLORS_VIS = {
    'No_Pain':(46,204,113),'Mild':(241,196,15),
    'Moderate':(230,126,34),'Severe':(231,76,60),
}

# ── Extract dataset ───────────────────────────────────────────────────────────
if not os.path.exists(DRIVE_ZIP):
    raise FileNotFoundError(f'ZIP not found: {DRIVE_ZIP}\nRun Cell 1 to see your Drive files.')

if WORK_DIR.exists():
    shutil.rmtree(WORK_DIR)
WORK_DIR.mkdir(parents=True)

print('Extracting dataset...')
with zipfile.ZipFile(DRIVE_ZIP, 'r') as z:
    z.extractall(WORK_DIR)
print('Extraction done.')

FRAMES_DIR = WORK_DIR / 'images' / 'extracted_frames'
EMO_DIR    = WORK_DIR / 'images' / 'Emotional_faces' / 'Emotional_faces'
print(f'extracted_frames : {FRAMES_DIR.exists()}')
print(f'Emotional_faces  : {EMO_DIR.exists()}')

# ── Collect all image paths with pain labels ──────────────────────────────────
all_samples = []

if FRAMES_DIR.exists():
    for label in CLASSES:
        d = FRAMES_DIR / label
        if d.exists():
            for p in d.iterdir():
                if p.suffix.lower() in IMG_EXTS:
                    all_samples.append((p, label))

if EMO_DIR.exists():
    for subj in EMO_DIR.iterdir():
        if not subj.is_dir(): continue
        for img in subj.iterdir():
            if img.suffix.lower() not in IMG_EXTS: continue
            pain = EMOTION_TO_PAIN.get(img.stem.lower())
            if pain:
                all_samples.append((img, pain))

counts = Counter(lbl for _, lbl in all_samples)
print(f'\nTotal images: {len(all_samples)}')
for cls in CLASSES:
    print(f'  {cls:<12} {counts[cls]}')

# ── Face detector ─────────────────────────────────────────────────────────────
face_cascade = cv2.CascadeClassifier(
    cv2.data.haarcascades + 'haarcascade_frontalface_default.xml'
)

def get_seg_label(image_path):
    img = cv2.imread(str(image_path))
    if img is None: return None
    h, w = img.shape[:2]
    gray = cv2.cvtColor(img, cv2.COLOR_BGR2GRAY)
    faces = face_cascade.detectMultiScale(gray, 1.1, 3, minSize=(20,20))
    if len(faces) > 0:
        fx, fy, fw, fh = faces[0]
        x1,y1,x2,y2 = fx, fy, fx+fw, fy+fh
    else:
        pad = int(min(h,w)*0.05)
        x1,y1,x2,y2 = pad,pad,w-pad,h-pad
    x1,y1 = max(0,x1),max(0,y1)
    x2,y2 = min(w,x2),min(h,y2)
    pts = [(x1/w,y1/h),(x2/w,y1/h),(x2/w,y2/h),(x1/w,y2/h)]
    return ' '.join(f'{x:.6f} {y:.6f}' for x,y in pts)

print('\nSetup complete. Ready to build dataset.')

In [ ]:
# ── STEP 4: Build YOLO segmentation dataset ───────────────────────────────────
from tqdm import tqdm

random.seed(42)

if YOLO_DIR.exists():
    shutil.rmtree(YOLO_DIR)
for split in ['train','val','test']:
    (YOLO_DIR/'images'/split).mkdir(parents=True)
    (YOLO_DIR/'labels'/split).mkdir(parents=True)

class_samples = {cls:[] for cls in CLASSES}
for path, label in all_samples:
    class_samples[label].append(path)

split_counts = Counter()
skipped = 0

for cls, paths in class_samples.items():
    random.shuffle(paths)
    n = len(paths)
    n_train = int(n * TRAIN_RATIO)
    n_val   = int(n * VAL_RATIO)
    splits = (
        [('train',p) for p in paths[:n_train]] +
        [('val',  p) for p in paths[n_train:n_train+n_val]] +
        [('test', p) for p in paths[n_train+n_val:]]
    )
    class_id = CLASS_TO_ID[cls]
    for split, src in tqdm(splits, desc=cls, leave=False):
        poly = get_seg_label(src)
        if poly is None:
            skipped += 1
            continue
        dst_img = YOLO_DIR/'images'/split/src.name
        if dst_img.exists():
            dst_img = dst_img.with_stem(dst_img.stem+f'_{cls[:3]}')
        shutil.copy2(src, dst_img)
        lbl = YOLO_DIR/'labels'/split/(dst_img.stem+'.txt')
        lbl.write_text(f'{class_id} {poly}\n')
        split_counts[f'{split}/{cls}'] += 1

print(f'Skipped: {skipped}')
for split in ['train','val','test']:
    total = sum(v for k,v in split_counts.items() if k.startswith(split))
    print(f'  {split:<6} {total} images')
    for cls in CLASSES:
        print(f'         {cls:<12} {split_counts[f"{split}/{cls}"]}')

In [ ]:
# ── STEP 5: Create data.yaml ──────────────────────────────────────────────────
yaml_content = f'path: {YOLO_DIR}\ntrain: images/train\nval:   images/val\ntest:  images/test\n\nnc: {len(CLASSES)}\nnames: {CLASSES}\n'
yaml_path = YOLO_DIR / 'data.yaml'
yaml_path.write_text(yaml_content)
print(yaml_content)

In [ ]:
# ── STEP 6: Verify — show sample images with mask polygons ───────────────────
import matplotlib.pyplot as plt
import matplotlib.patches as mpatches
from PIL import Image

train_imgs = list((YOLO_DIR/'images'/'train').iterdir())[:8]
fig, axes = plt.subplots(2, 4, figsize=(16,8))
fig.patch.set_facecolor('#0f1117')

for ax, img_path in zip(axes.flat, train_imgs):
    lbl_path = YOLO_DIR/'labels'/'train'/(img_path.stem+'.txt')
    img_np = np.array(Image.open(img_path).convert('RGB'))
    h, w = img_np.shape[:2]
    overlay = img_np.copy()
    ax.set_facecolor('#0f1117'); ax.axis('off')
    if lbl_path.exists():
        for line in lbl_path.read_text().strip().splitlines():
            parts = list(map(float, line.split()))
            cid = int(parts[0])
            cls_name = ID_TO_CLASS[cid]
            color = COLORS_VIS[cls_name]
            pts = np.array(parts[1:], dtype=np.float32).reshape(-1,2)
            pts[:,0] *= w; pts[:,1] *= h
            pts = pts.astype(np.int32)
            mask_img = np.zeros_like(img_np)
            cv2.fillPoly(mask_img, [pts], color)
            overlay = cv2.addWeighted(overlay, 0.6, mask_img, 0.4, 0)
            cv2.polylines(overlay, [pts], True, color, 2)
            ax.set_title(cls_name, color='#{:02x}{:02x}{:02x}'.format(*color), fontsize=9, fontweight='bold')
    ax.imshow(overlay)

plt.suptitle('Sample Training Images with Segmentation Masks', color='white', fontsize=13)
plt.tight_layout()
plt.savefig('/content/sample_masks.png', dpi=100, bbox_inches='tight', facecolor='#0f1117')
plt.show()
print('Saved: /content/sample_masks.png')

In [ ]:
# ── STEP 7: Train YOLOv8n-seg ─────────────────────────────────────────────────
from ultralytics import YOLO
import torch

print(f'CUDA: {torch.cuda.is_available()}')
if torch.cuda.is_available():
    print(f'GPU : {torch.cuda.get_device_name(0)}')

model = YOLO('yolov8n-seg.pt')
results = model.train(
    data     = str(yaml_path),
    epochs   = 50,
    imgsz    = 224,
    batch    = 16,
    device   = 0 if torch.cuda.is_available() else 'cpu',
    project  = str(RESULTS_DIR),
    name     = 'pain_seg',
    patience = 10,
    exist_ok = True,
    plots    = True,
)

BEST_WEIGHTS = RESULTS_DIR / 'pain_seg' / 'weights' / 'best.pt'
print(f'\nDone. Best weights: {BEST_WEIGHTS}')

In [ ]:
# ── STEP 8: Evaluate — Mask mAP ───────────────────────────────────────────────
best_model = YOLO(str(BEST_WEIGHTS))
metrics = best_model.val(
    data=str(yaml_path), split='test', imgsz=224,
    device=0 if torch.cuda.is_available() else 'cpu',
    plots=True, save_json=True,
)

print('='*50)
print('  Week 4 — Segmentation Results')
print('='*50)
print(f'  Box  mAP@0.5      : {metrics.box.map50*100:.2f}%')
print(f'  Mask mAP@0.5      : {metrics.seg.map50*100:.2f}%')
print(f'  Mask mAP@0.5:0.95 : {metrics.seg.map*100:.2f}%')
print(f'  Mask Precision    : {metrics.seg.mp*100:.2f}%')
print(f'  Mask Recall       : {metrics.seg.mr*100:.2f}%')
print('\nPer-class Mask AP@0.5:')
for cls, ap in zip(CLASSES, metrics.seg.ap50):
    print(f'  {cls:<12} {ap*100:.2f}%')

seg_results = {
    'box_mAP50':     round(float(metrics.box.map50),4),
    'mask_mAP50':    round(float(metrics.seg.map50),4),
    'mask_mAP50_95': round(float(metrics.seg.map),4),
    'mask_precision':round(float(metrics.seg.mp),4),
    'mask_recall':   round(float(metrics.seg.mr),4),
    'per_class_mask_AP50': {cls: round(float(ap),4) for cls,ap in zip(CLASSES,metrics.seg.ap50)}
}
with open('/content/seg_results.json','w') as f:
    json.dump(seg_results, f, indent=2)
print('\nSaved: /content/seg_results.json')

In [ ]:
# ── STEP 9: Predict on test images ────────────────────────────────────────────
best_model.predict(
    source=str(YOLO_DIR/'images'/'test'),
    imgsz=224, conf=0.25, save=True,
    project='/content/seg_predictions', name='test_preds', exist_ok=True,
    device=0 if torch.cuda.is_available() else 'cpu',
)
print('Predictions saved.')

In [ ]:
# ── STEP 10: Plot prediction grid ─────────────────────────────────────────────
pred_imgs = sorted(Path('/content/seg_predictions/test_preds').glob('*.jpg'))[:12]
n = len(pred_imgs); cols=4; rows=(n+cols-1)//cols
fig, axes = plt.subplots(rows, cols, figsize=(16, rows*4))
fig.patch.set_facecolor('#0f1117')
for ax, p in zip(axes.flat, pred_imgs):
    ax.imshow(np.array(Image.open(p))); ax.axis('off')
    ax.set_title(p.name[:18], color='white', fontsize=7)
for ax in axes.flat[n:]: ax.set_visible(False)
plt.suptitle('YOLOv8-seg Predictions', color='white', fontsize=13)
plt.tight_layout()
plt.savefig('/content/seg_prediction_grid.png', dpi=100, bbox_inches='tight', facecolor='#0f1117')
plt.show()

In [ ]:
# ── STEP 11: Save all results to Google Drive ─────────────────────────────────
DRIVE_OUT = Path('/content/drive/MyDrive/Week4_Results')
DRIVE_OUT.mkdir(parents=True, exist_ok=True)

shutil.copy2(BEST_WEIGHTS, DRIVE_OUT/'best_seg.pt')
shutil.copy2('/content/seg_results.json', DRIVE_OUT/'seg_results.json')

for fname in ['sample_masks.png','seg_prediction_grid.png']:
    src = Path(f'/content/{fname}')
    if src.exists(): shutil.copy2(src, DRIVE_OUT/fname)

yolo_res = RESULTS_DIR/'pain_seg'
if yolo_res.exists():
    shutil.copytree(yolo_res, DRIVE_OUT/'yolo_seg_results', dirs_exist_ok=True)

pred_out = DRIVE_OUT/'prediction_samples'
pred_out.mkdir(exist_ok=True)
for p in sorted(Path('/content/seg_predictions/test_preds').glob('*.jpg'))[:20]:
    shutil.copy2(p, pred_out/p.name)

print('All results saved to Google Drive/Week4_Results/')